# Qwen2.5-VL-3B 마케팅 JSON 파인튜닝 — Colab 버전

## 🚀 실행 순서
1. **런타임 → 런타임 유형 변경 → T4 GPU 선택** (필수!)
2. 아래 셀을 **Shift + Enter** 로 하나씩 실행
3. 중간에 파일 업로드 셀에서 `images.zip`과 `data_list.jsonl` 올리기
4. 학습 끝나면 `lora_output.zip` 자동 다운로드

## 📦 미리 준비할 것
- **`data_list.jsonl`** — JSONL 파일
- **`images.zip`** — 이미지 100장을 압축한 zip 파일
  - Mac: `images/` 폴더 우클릭 → "images을(를) 압축하기" → `images.zip` 생성됨
  - Windows: `images` 폴더 우클릭 → "압축(ZIP) 폴더로 보내기"

예상 소요 시간: **1~2시간**


## 1. GPU 확인


In [ ]:
!nvidia-smi

## 2. 패키지 설치 (2~3분)


In [ ]:
!pip install -q "transformers>=4.49" accelerate peft bitsandbytes qwen-vl-utils

## 3. 파일 업로드

**아래 셀 실행 → "파일 선택" 버튼 눌러서 두 파일 함께 선택**:
- `data_list.jsonl`
- `images.zip`


In [ ]:
from google.colab import files
import os, zipfile

# 파일 업로드
uploaded = files.upload()

# images.zip 압축 풀기
for name in uploaded:
    if name.endswith(".zip"):
        print(f"\n{name} 압축 해제 중...")
        with zipfile.ZipFile(name, "r") as z:
            z.extractall(".")
        print("완료!")

# 확인
print("\n업로드된 파일:")
for f in os.listdir("."):
    if not f.startswith(".") and f not in ["sample_data"]:
        print(" -", f)

# images 폴더 확인
img_dir = None
for cand in ["images", "./images"]:
    if os.path.isdir(cand):
        img_dir = cand
        break

if img_dir:
    imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(".png")]
    print(f"\n이미지 {len(imgs)}장 발견 ({img_dir})")
else:
    print("\n⚠️ images 폴더가 없어요! zip이 제대로 풀렸는지 확인하세요.")

## 4. 라이브러리 import 및 설정


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, json, random
import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID   = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGE_DIR  = "/content/drive/MyDrive/g/images"
JSONL_PATH = "/content/drive/MyDrive/g/code/data_list.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/g/qwen_vl_lora"

MAX_SIDE   = 512
MAX_PIXELS = 512 * 512
EVAL_RATIO = 0.1
SEED       = 42
EPOCHS     = 3

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"DTYPE: {COMPUTE_DTYPE}, GPU: {torch.cuda.get_device_name(0)}")

random.seed(SEED); torch.manual_seed(SEED)

## 5. 데이터 로드 (JSONL → data_list)


In [ ]:
data_list = []
system_prompt = None
with open(JSONL_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        entry = json.loads(line)
        messages = entry["messages"]
        sys_msg = next(m for m in messages if m["role"] == "system")
        usr_msg = next(m for m in messages if m["role"] == "user")
        ast_msg = next(m for m in messages if m["role"] == "assistant")

        if system_prompt is None:
            system_prompt = sys_msg["content"]

        image_path, user_text = None, ""
        for c in usr_msg["content"]:
            if c["type"] == "image":  image_path = c["image"]
            elif c["type"] == "text": user_text = c["text"]

        assistant_json = json.loads(ast_msg["content"])
        data_list.append({
            "image": image_path,
            "user_text": user_text,
            "assistant_json": assistant_json,
        })

print(f"JSONL 로드: {len(data_list)}개")

# 이미지 파일 존재 확인
samples, skipped = [], []
for item in data_list:
    img_name = os.path.basename(item["image"])
    # .PNG → .jpg 로 자동 변환
    img_name = img_name.replace(".PNG", ".jpg").replace(".png", ".jpg")
    img_path = os.path.join(IMAGE_DIR, img_name)
    if not os.path.exists(img_path):
        skipped.append(img_name); continue
    samples.append({
        "image_path": img_path,
        "user_text": item["user_text"],
        "target_json": json.dumps(item["assistant_json"], ensure_ascii=False, indent=2),
    })

print(f"유효 샘플: {len(samples)} / 이미지 없음: {len(skipped)}")
if skipped:
    print("누락된 이미지 (첫 5개):", skipped[:5])

random.shuffle(samples)
n_eval = max(1, int(len(samples) * EVAL_RATIO))
eval_samples, train_samples = samples[:n_eval], samples[n_eval:]
print(f"train: {len(train_samples)} / eval: {len(eval_samples)}")

class VLMDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

train_ds = VLMDataset(train_samples)
eval_ds  = VLMDataset(eval_samples)

## 6. 이미지 전처리


In [ ]:
def preprocess_image(path):
    img = Image.open(path)
    img.thumbnail((MAX_SIDE, MAX_SIDE))
    if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
        img = img.convert("RGBA")
        gray_bg = Image.new("RGB", img.size, (128, 128, 128))
        gray_bg.paste(img, mask=img.split()[3])
        return gray_bg
    return img.convert("RGB")

## 7. 모델 로드 (2~3분)


In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    llm_int8_skip_modules=["visual", "lm_head"],
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=r"^(?!.*visual).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. 배치 collator


In [ ]:
def build_messages(user_text, target_json=None):
    msgs = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": user_text},
        ]},
    ]
    if target_json is not None:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": target_json}]})
    return msgs

ASSISTANT_START = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

def find_last(seq, sub):
    for i in range(len(seq) - len(sub), -1, -1):
        if seq[i:i + len(sub)] == sub:
            return i
    return -1

def collate_fn(batch):
    texts, images = [], []
    for ex in batch:
        images.append(preprocess_image(ex["image_path"]))
        texts.append(processor.apply_chat_template(
            build_messages(ex["user_text"], ex["target_json"]),
            tokenize=False, add_generation_prompt=False,
        ))
    enc = processor(text=texts, images=images, padding=True, return_tensors="pt")
    labels = enc["input_ids"].clone()
    labels[enc["attention_mask"] == 0] = -100
    for i in range(labels.size(0)):
        start = find_last(enc["input_ids"][i].tolist(), ASSISTANT_START)
        if start == -1:
            labels[i, :] = -100
        else:
            labels[i, : start + len(ASSISTANT_START)] = -100
    enc["labels"] = labels
    return enc

# 검증
_check = collate_fn([train_samples[0]])
print(f"학습 대상 토큰: {(_check['labels'][0] != -100).sum().item()}")

## 9. 🔥 학습 시작 (1~2시간)


In [ ]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=5,
    warmup_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=(COMPUTE_DTYPE == torch.float16),
    bf16=(COMPUTE_DTYPE == torch.bfloat16),
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=collate_fn,
)
trainer.train()

trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"\n✅ 학습 완료! 저장: {OUTPUT_DIR}")

## 10. 추론 테스트


In [ ]:
model.eval()
model.config.use_cache = True

def extract_json(text):
    text = text.replace("```json", "").replace("```", "")
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m: return None
    try: return json.loads(m.group(0))
    except: return None

@torch.no_grad()
def generate(image_path, user_text):
    img = preprocess_image(image_path)
    prompt = processor.apply_chat_template(
        build_messages(user_text), tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[img], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]

for ex in eval_samples[:3]:
    raw = generate(ex["image_path"], ex["user_text"])
    parsed = extract_json(raw)
    print("=" * 60)
    print("입력:", ex["user_text"] or "(빈 입력)")
    print("JSON 파싱:", "✅ 성공" if parsed else "❌ 실패")
    if parsed:
        print(json.dumps(parsed, ensure_ascii=False, indent=2))
    else:
        print(raw[:500])

## 11. LoRA 어댑터 다운로드


In [ ]:
import shutil
shutil.make_archive("lora_output", "zip", OUTPUT_DIR)
print("압축 완료: lora_output.zip")

from google.colab import files
files.download("lora_output.zip")
print("다운로드 시작! 로컬에 저장됩니다.")